In [3]:
# Following example from https://www.datacamp.com/tutorial/pytorch-cnn-tutorial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torchmetrics
import os
import cv2

In [4]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(300 * 300, num_classes) # resolution size
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        return x

In [5]:
cwd = os.getcwd()
data_path = os.path.join(cwd, "Data")
train_path = os.path.join(data_path, "train")
test_path = os.path.join(data_path, "test")

transform = transforms.Compose([
    transforms.ToTensor()
])

print("Setting up datasets...")
training_dataset = datasets.ImageFolder(root=train_path, transform=transform)
testing_dataset = datasets.ImageFolder(root=test_path, transform=transform)

train_loader = DataLoader(training_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(testing_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Done! Found {len(training_dataset)} training images and {len(testing_dataset)} testing images.")
print(f"Class mapping: {training_dataset.class_to_idx}")

Setting up datasets...
Done! Found 104000 training images and 20800 testing images.
Class mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25}


In [6]:
batch_size = 60

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN(in_channels=3, num_classes=26).to(device)
print(model)

CNN(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=90000, out_features=26, bias=True)
)


In [ ]:
# Training the CNN
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epoch = 1
for epoch in range(num_epoch):
    print(f"Epoch [{epoch + 1}/{num_epoch}]")
    for batch_index, (data, targets) in enumerate(tqdm(train_loader)):
        data = data.to(device)
        targets = targets.to(device)
        scores = model(data)
        loss = criterion(scores, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


torch.save(model.state_dict(), 'MulticlassCNN.pth')

Epoch [1/1]


  0%|          | 10/3250 [00:20<1:49:20,  2.02s/it]


In [8]:
# Evaluate the CNN
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall

loaded_model = CNN(in_channels=3, num_classes=26).to(device)
loaded_model.load_state_dict(torch.load('MulticlassCNN.pth', map_location=device, weights_only=True))

metric_collection = MetricCollection({
    "accuracy": MulticlassAccuracy(num_classes=26, average="micro"),
    "precision": MulticlassPrecision(num_classes=26, average="macro"),
    "recall": MulticlassRecall(num_classes=26, average="macro")
}).to(device)

loaded_model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = loaded_model(images)
        _, preds = torch.max(outputs, 1)
        
        metric_collection.update(preds, labels)

results = metric_collection.compute()
print(f"Test accuracy: {results['accuracy']:.4f}")
print(f"Test precision: {results['precision']:.4f}")
print(f"Test recall: {results['recall']:.4f}")


Test accuracy: 0.5773
Test precision: 0.5544
Test recall: 0.5773
